# Lab 03 · The tool loop, including the failure path
**~30 minutes · costs about $0.03 · Domain 8 (10.6%), Domain 1 Agent Construction**

Proves the fact that underpins the whole security model: **the model never
executes anything.** It emits a request; your harness decides.

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## A tool, and a real implementation behind it

In [ ]:
ORDERS = {"CUST-10041": "shipped", "CUST-10042": "processing"}

TOOLS = [{
    "name": "lookup_order",
    "description": ("Look up the status of a customer order by ID. "
                    "Use when the user asks about an order's status or delivery. "
                    "Do NOT use for refunds or cancellations."),
    "input_schema": {
        "type":"object",
        "properties":{"customer_id":{"type":"string",
                      "description":"Customer ID in the form CUST-XXXXX"}},
        "required":["customer_id"],
    },
}]

def run_tool(name, args):
    if name != "lookup_order":
        raise ValueError(f"unknown tool {name}")
    cid = args["customer_id"]
    if cid not in ORDERS:
        raise KeyError(f"customer_id {cid} not found; expected format CUST-XXXXX")
    return ORDERS[cid]

## The loop, bounded

Note `MAX_TURNS`. An unbounded loop is a harness design failure, not a model
failure, and the exam asks about it directly.

In [ ]:
def agent(question, max_turns=6, verbose=True):
    msgs = [{"role":"user","content":question}]
    for turn in range(max_turns):
        r = client.messages.create(model=MODEL, max_tokens=800, tools=TOOLS, messages=msgs)
        if verbose: print(f"turn {turn}: stop_reason={r.stop_reason}")
        if r.stop_reason != "tool_use":
            return "".join(b.text for b in r.content if b.type=="text")
        msgs.append({"role":"assistant","content":r.content})
        results = []
        for b in r.content:
            if b.type == "tool_use":
                if verbose: print("   model asked for", b.name, b.input)
                try:
                    results.append({"type":"tool_result","tool_use_id":b.id,
                                    "content":run_tool(b.name, b.input)})
                except Exception as e:
                    if verbose: print("   TOOL FAILED ->", e)
                    results.append({"type":"tool_result","tool_use_id":b.id,
                                    "content":str(e),"is_error":True})
        msgs.append({"role":"user","content":results})
    return "HIT THE TURN LIMIT"

print(agent("What is the status of order CUST-10041?"))

## The important half: a deliberate failure

Watch the model receive `is_error: True`, read the message, and **self-correct**.
This is why you return actionable errors instead of throwing.

In [ ]:
print(agent("Check order 10042 for me."))

## Now make tool selection worse

Add an overlapping tool with a vague description. The model has to guess, and
guesses badly. Every extra tool also costs tokens on **every** request.

In [ ]:
TOOLS.append({
    "name":"get_customer_info",
    "description":"Gets information about a customer.",      # vague on purpose
    "input_schema":{"type":"object",
        "properties":{"id":{"type":"string"}},"required":["id"]},
})

base = client.messages.count_tokens(model=MODEL, tools=TOOLS[:1],
        messages=[{"role":"user","content":"status of CUST-10041?"}]).input_tokens
both = client.messages.count_tokens(model=MODEL, tools=TOOLS,
        messages=[{"role":"user","content":"status of CUST-10041?"}]).input_tokens
print(f"1 tool: {base} tokens | 2 tools: {both} tokens | overhead every request: +{both-base}")
print()
print(agent("Tell me about CUST-10041."))

---
### Checkpoint
- Who executes a tool?
- Why is a tool description a prompt?
- What do you return when a tool throws, and why not just raise?
- Why do 40 tools make selection worse rather than better?